# 04 — Pipeline complet avec la voix clonée XTTS-v2 (Colab)

Vidéo française → Whisper → traduction → **relecture** → voix clonée XTTS-v2 → synchronisation → vidéo.

Pourquoi Colab : XTTS-v2 (paquet `coqui-tts`) ne s'installe pas sur le Mac Intel du projet
(dépendance `monotonic-alignment-search` sans version pour macOS x86_64).

**À préparer :**
- une vidéo française courte (créée par l'équipe ou libre de droits) ;
- un enregistrement de **ta propre voix** (10 à 30 s, au calme), ou d'une personne qui a donné son accord écrit.

Téléchargements dans Colab : XTTS-v2 ≈ 2,1 Go, traduction ≈ 0,9 Go, Whisper base ≈ 0,15 Go.
Fonctionne sur CPU (lent : compter plusieurs minutes par minute de vidéo) ; un GPU accélère la voix.

## 1. Récupérer le projet et installer

In [ ]:
!git clone -b abdou https://github.com/Abdouaziz2/ai-video-dubbing.git
%cd ai-video-dubbing
!pip install -q -r requirements.txt -r requirements-voice.txt

In [ ]:
!python -m pytest -q

In [ ]:
import sys, torch, transformers, TTS
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| GPU", torch.cuda.is_available(),
      "| transformers", transformers.__version__, "| coqui-tts", TTS.__version__)

## 2. Licence XTTS-v2 et consentement

Licence du modèle : **Coqui Public Model License**, usage **non commercial** uniquement —
https://coqui.ai/cpml.txt (la lire avant d'accepter).

In [ ]:
import os
I_ACCEPT_CPML = False        # True seulement après avoir lu la licence
CONSENT_CONFIRMED = False    # True seulement si la personne enregistrée a donné son accord écrit
if not (I_ACCEPT_CPML and CONSENT_CONFIRMED):
    raise RuntimeError("Lis la licence et confirme le consentement écrit avant de continuer.")
os.environ["COQUI_TOS_AGREED"] = "1"

## 3. Téléverser la vidéo et la voix de référence

In [ ]:
from google.colab import files
print("Choisis la VIDÉO française :")
VIDEO = list(files.upload())[0]
print("Choisis l'ENREGISTREMENT de la voix de référence :")
raw_voice = list(files.upload())[0]

import subprocess
REFERENCE = "reference_voice.wav"   # conversion en WAV mono, format le plus sûr pour XTTS
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw_voice, "-ac", "1", "-ar", "22050", REFERENCE],
               check=True)
print("Vidéo :", VIDEO, "| référence :", REFERENCE)

## 4a. Option rapide : tout le pipeline sans relecture

Le résultat va dans `data/output/colab_run/`.

In [ ]:
!python -m src.pipeline --video "$VIDEO" --voice xtts --reference "$REFERENCE" --output-dir data/output/colab_run

In [ ]:
from IPython.display import Video
Video("data/output/colab_run/dubbed_video.mp4", embed=True, width=640)

## 4b. Option avec relecture : l'interface Gradio

`share=True` crée un lien public temporaire (72 h) vers l'interface : ne le diffuse pas, toute
personne qui a le lien peut utiliser l'application.
Dans l'interface, choisis la voix **XTTS-v2**, ajoute `reference_voice.wav` et coche la case de consentement.

In [ ]:
from app.gradio_app import demo
demo.launch(share=True, show_error=True, debug=True)

## 5. Récupérer les résultats

La vidéo doublée, le détail par segment (`final_segments.json`) et les audios par segment
(utiles pour le test d'écoute à l'aveugle, voir `docs/evaluation_protocol.md`).

In [ ]:
import json, shutil
from src.evaluation import summarize_durations
run_dir = "data/output/colab_run"   # ou le dossier data/output/run_<date> créé par l'interface
print(summarize_durations(json.load(open(f"{run_dir}/final_segments.json"))["segments"]))
shutil.make_archive("resultats_doublage", "zip", run_dir)
files.download("resultats_doublage.zip")